In [ ]:
import numpy as np
from pathlib import Path
import json

# Define sequence segmentation method
def segment_sequence(sequence_length, keypoints):
    num_frames = len(keypoints)
    clips = []
    
    # Sliding window - only create complete clips
    for start_idx in range(0, num_frames - sequence_length + 1, sequence_length):
        end_idx = start_idx + sequence_length
        clip = keypoints[start_idx:end_idx]
        
        # Verify clip has correct length
        assert len(clip) == sequence_length, f"Clip has wrong length: {len(clip)}"
        
        clips.append(clip)
    
    return clips

# MAIN
input_dir = "data/extracted_keypoints"
output_dir = "data/segmented_clips"
input_path = Path(input_dir)
output_path = Path(output_dir)
output_path.mkdir(parents=True, exist_ok=True)
    
# Configuration
sequence_length = 60  # 60 frames = 2 seconds at 30fps

# Load metadata from extraction step
metadata_file = input_path / 'extraction_metadata.json'
if not metadata_file.exists():
    raise ValueError(f"Metadata file not found: {metadata_file}")

with open(metadata_file, 'r') as f:
    extraction_metadata = json.load(f)

print(f"Found {len(extraction_metadata)} keypoint sequences to segment\n")

all_clips_metadata = []
total_clips = 0
discarded_frames = 0
total_frames = 0

# Iterate through videos
for i, (video_name, info) in enumerate(extraction_metadata.items(), 1):
    print(f"[{i}/{len(extraction_metadata)}] {video_name}")
    
    # Load keypoints
    keypoints_file = info['keypoints_file']
    keypoints = np.load(keypoints_file)
    
    num_frames = len(keypoints)
    total_frames += num_frames
    
    # Segment into clips
    clips = segment_sequence(sequence_length, keypoints)
    
    # Calculate discarded frames
    frames_used = len(clips) * sequence_length
    frames_discarded = num_frames - frames_used
    discarded_frames += frames_discarded
    
    print(f"  {num_frames} frames → {len(clips)} clips "
          f"({frames_discarded} frames discarded)")
    
    # Save each clip
    for clip_idx, clip in enumerate(clips):
        # Create output filename
        clip_filename = output_path / f"{video_name}_clip{clip_idx:03d}.npy"
        
        # Create parent directories if needed
        clip_filename.parent.mkdir(parents=True, exist_ok=True)
        
        # Save clip
        np.save(clip_filename, clip)
        
        # Store metadata for this clip
        all_clips_metadata.append({
            'clip_file': str(clip_filename),
            'original_video': video_name,
            'original_video_category': info['category'],
            'clip_index': clip_idx,
            'num_clips_from_video': len(clips),
            'sequence_length': sequence_length,
            'start_frame': clip_idx * sequence_length,
            'end_frame': (clip_idx * sequence_length) + sequence_length,
            'clip_shape': list(clip.shape),
            'width': info['width'],
            'height': info['height'],
            'resolution': info['resolution']
        })
        
        total_clips += 1
    
    if i % 100 == 0:
        print(f"\n--- Progress: {i}/{len(extraction_metadata)} ---")
        print(f"    Clips created so far: {total_clips}\n")

# Save clips metadata
clips_metadata_file = output_path / 'clips_metadata.json'
with open(clips_metadata_file, 'w') as f:
    json.dump(all_clips_metadata, f, indent=2)

# Print summary
print("\n" + "=" * 70)
print("SEGMENTATION COMPLETE")
print("=" * 70)
print(f"Total videos processed: {len(extraction_metadata)}")
print(f"Total clips created: {total_clips}")
print(f"Average clips per video: {total_clips/len(extraction_metadata):.1f}")
print(f"\nFrame usage:")
print(f"  Total frames: {total_frames}")
print(f"  Frames in clips: {total_clips * sequence_length}")
print(f"  Frames discarded: {discarded_frames} ({discarded_frames/total_frames*100:.1f}%)")
print(f"\nMetadata saved to: {clips_metadata_file}")
print(f"Clips saved to: {output_path}")